# 03. Star Schema — Database Validation

## Purpose

The relational star schema has already been created and checked in DBeaver.
This notebook does **not** rebuild it. Instead, it connects to the existing
DuckDB database, verifies that the expected tables and keys are present,
and produces a compact set of validation outputs.

This keeps the SQL implementation in `sql/02_create_star_schema.sql` as
the single source of truth, while making the database checks reproducible
from Python.

## 1. Import libraries and locate the project

In [9]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------

current_path = Path.cwd()

# Locate the project root by searching for data/processed.
# This follows the same approach used in notebook 02.
PROJECT_ROOT = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing data/processed. "
        "Please check where Jupyter is running."
    )

# Define the main project directories
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "reports" / "results"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

# Database path
DB_PATH = PROCESSED_DIR / "seizure_signals.duckdb"

# Create output directories if they do not exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Check that the database exists
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found: {DB_PATH}"
    )

print("Project root:", PROJECT_ROOT)
print("Database:", DB_PATH)
print("Results directory:", RESULTS_DIR)
print("Figures directory:", FIGURES_DIR)

Project root: C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals
Database: C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals\data\processed\seizure_signals.duckdb
Results directory: C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals\reports\results
Figures directory: C:\Users\joaqu\OneDrive\Escritorio\CV\Postulaciones\XX. UNIDO\Personal Project\GitHub\seizure-signals\reports\figures


In [10]:
# Connect to the existing DuckDB database.
# read_only=True prevents accidental modifications.
con = duckdb.connect(
    database=str(DB_PATH),
    read_only=True
)

print("Connected to DuckDB successfully.")

Connected to DuckDB successfully.


## 2. Check that the expected tables exist

In [11]:
# Tables expected from the SQL star-schema build.
expected_tables = [
    "ids_staging",
    "drug_reference",
    "dim_date",
    "dim_country",
    "dim_drug",
    "fact_seizure",
]

available_tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
      AND table_type = 'BASE TABLE'
    ORDER BY table_name
""").df()

display(available_tables)

available = set(available_tables["table_name"])
missing_tables = sorted(set(expected_tables) - available)

if missing_tables:
    raise RuntimeError(
        "The following expected tables are missing: "
        + ", ".join(missing_tables)
        + ". Check that SQL/02_create_star_schema.sql was executed."
    )

print("All expected tables are present.")

,table_name
0,dim_country
1,dim_date
2,dim_drug
3,drug_reference
4,fact_seizure
5,ids_staging


All expected tables are present.


## 3. Compare row counts across staging, fact, and dimensions

In [12]:
# A fact row should correspond to one original source row.
# The fact table should therefore have the same number of records
# as ids_staging, assuming the complete ETL ran successfully.

table_names = [
    "ids_staging",
    "drug_reference",
    "dim_date",
    "dim_country",
    "dim_drug",
    "fact_seizure",
]

row_counts = []
for table_name in table_names:
    count = con.execute(
        f'SELECT COUNT(*) AS n FROM "{table_name}"'
    ).fetchone()[0]
    row_counts.append({"table": table_name, "rows": count})

row_counts = pd.DataFrame(row_counts)
display(row_counts)

staging_rows = int(
    row_counts.loc[row_counts["table"] == "ids_staging", "rows"].iloc[0]
)
fact_rows = int(
    row_counts.loc[row_counts["table"] == "fact_seizure", "rows"].iloc[0]
)

print("Staging records:", f"{staging_rows:,}")
print("Fact records:", f"{fact_rows:,}")
print("Difference:", f"{staging_rows - fact_rows:,}")

if staging_rows != fact_rows:
    print("WARNING: staging and fact row counts differ.")
else:
    print("PASS: staging and fact row counts match.")

,table,rows
0,ids_staging,2841288
1,drug_reference,184
2,dim_date,5844
3,dim_country,147
4,dim_drug,345
5,fact_seizure,2841288


Staging records: 2,841,288
Fact records: 2,841,288
Difference: 0
PASS: staging and fact row counts match.


## 4. Inspect table structures

In [13]:
# Review the columns and data types in the fact table and dimensions.
for table_name in [
    "fact_seizure", "dim_date", "dim_country", "dim_drug"
]:
    print(f"\n{table_name}")
    schema = con.execute(
        f"DESCRIBE SELECT * FROM {table_name}"
    ).df()
    display(schema)


fact_seizure


,column_name,column_type,null,key,default,extra
0,seizure_key,BIGINT,NO,PRI,None,None
1,date_key,DATE,YES,None,None,None
2,country_key,BIGINT,YES,None,None,None
3,drug_key,BIGINT,YES,None,None,None
4,quantity_seized,DOUBLE,YES,None,None,None
5,measurement_unit,VARCHAR,YES,None,None,None
6,city,VARCHAR,YES,None,None,None
7,administrative_region,VARCHAR,YES,None,None,None
8,physical_seizure_location,VARCHAR,YES,None,None,None
9,transportation_mode,VARCHAR,YES,None,None,None



dim_date


,column_name,column_type,null,key,default,extra
0,date_key,DATE,NO,PRI,None,None
1,year,BIGINT,YES,None,None,None
2,month,BIGINT,YES,None,None,None
3,month_name,VARCHAR,YES,None,None,None
4,quarter,BIGINT,YES,None,None,None
5,day,BIGINT,YES,None,None,None
6,day_of_week,BIGINT,YES,None,None,None
7,day_name,VARCHAR,YES,None,None,None
8,is_weekend,BOOLEAN,YES,None,None,None



dim_country


,column_name,column_type,null,key,default,extra
0,country_key,BIGINT,NO,PRI,None,None
1,ISO3,VARCHAR,NO,UNI,None,None
2,country_name,VARCHAR,YES,None,None,None



dim_drug


,column_name,column_type,null,key,default,extra
0,drug_key,BIGINT,NO,PRI,None,None
1,drug_name,VARCHAR,NO,UNI,None,None
2,drug_class,VARCHAR,YES,None,None,None
3,drug_subgroup,VARCHAR,YES,None,None,None


## 5. Validate primary keys and foreign-key matches

In [14]:
# Check that the surrogate keys are unique in each dimension and fact.
key_checks = {
    "fact_seizure": "seizure_key",
    "dim_date": "date_key",
    "dim_country": "country_key",
    "dim_drug": "drug_key",
}

key_results = []
for table_name, key_column in key_checks.items():
    result = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT {key_column}) AS unique_keys,
            SUM(CASE WHEN {key_column} IS NULL THEN 1 ELSE 0 END)
                AS null_keys
        FROM {table_name}
    """).fetchone()

    key_results.append({
        "table": table_name,
        "total_rows": result[0],
        "unique_keys": result[1],
        "null_keys": result[2],
        "duplicate_keys": result[0] - result[1] - result[2],
    })

key_results = pd.DataFrame(key_results)
display(key_results)

# Every fact foreign key should match a row in its dimension.
orphan_checks = con.execute("""
    SELECT
        SUM(CASE WHEN dd.date_key IS NULL THEN 1 ELSE 0 END)
            AS missing_date_matches,
        SUM(CASE WHEN dc.country_key IS NULL THEN 1 ELSE 0 END)
            AS missing_country_matches,
        SUM(CASE WHEN dg.drug_key IS NULL THEN 1 ELSE 0 END)
            AS missing_drug_matches
    FROM fact_seizure f
    LEFT JOIN dim_date dd
        ON f.date_key = dd.date_key
    LEFT JOIN dim_country dc
        ON f.country_key = dc.country_key
    LEFT JOIN dim_drug dg
        ON f.drug_key = dg.drug_key
""").df()

display(orphan_checks)

,table,total_rows,unique_keys,null_keys,duplicate_keys
0,fact_seizure,2841288,2841288,0,0
1,dim_date,5844,5844,0,0
2,dim_country,147,147,0,0
3,dim_drug,345,345,0,0


,missing_date_matches,missing_country_matches,missing_drug_matches
0,0.0,0.0,0.0


## 6. Annual fact-table coverage

In [15]:
# source_year refers to the year of the source workbook sheet.
# It is not necessarily identical to the calendar year of seizure_date.

annual_counts = con.execute("""
    SELECT
        source_year,
        COUNT(*) AS records,
        COUNT(DISTINCT country_key) AS reporting_countries,
        COUNT(DISTINCT drug_key) AS distinct_drugs
    FROM fact_seizure
    GROUP BY source_year
    ORDER BY source_year
""").df()

display(annual_counts)

annual_counts.to_csv(
    RESULTS_DIR / "star_schema_annual_validation.csv",
    index=False
)
print("Saved annual validation output.")

,source_year,records,reporting_countries,distinct_drugs
0,2011,41806,67,101
1,2012,62090,72,143
2,2013,62036,63,139
3,2014,265457,53,175
4,2015,238294,45,140
5,2016,307817,46,145
6,2017,232409,54,107
7,2018,220003,82,112
8,2019,119368,77,106
9,2020,151682,84,120


Saved annual validation output.


## 7. Final validation summary

In [16]:
# Compact summary of the database build.
summary = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM ids_staging) AS staging_records,
        (SELECT COUNT(*) FROM fact_seizure) AS fact_records,
        (SELECT COUNT(*) FROM dim_date) AS dates,
        (SELECT COUNT(*) FROM dim_country) AS countries,
        (SELECT COUNT(*) FROM dim_drug) AS drugs,
        (SELECT COUNT(*) FROM drug_reference) AS reference_drugs
""").df()

display(summary)

print(
    "Review the row-count, key-uniqueness, and foreign-key checks above. "
    "A successful notebook run is not by itself proof that every data "
    "quality issue has been resolved."
)

,staging_records,fact_records,dates,countries,drugs,reference_drugs
0,2841288,2841288,5844,147,345,184


Review the row-count, key-uniqueness, and foreign-key checks above. A successful notebook run is not by itself proof that every data quality issue has been resolved.


## Conclusions

Complete this section after running the notebook.

Record whether:
- all expected tables exist;
- staging and fact row counts match;
- dimension keys are unique and non-null;
- fact foreign keys all match their dimensions;
- annual record counts match the results already validated in DBeaver.

The star schema is a structured representation of the source records.
`seizure_key` is a technical surrogate key, not an official IDS event ID.

In [17]:
con.close()
print('DuckDB connection closed.')

DuckDB connection closed.
